# Introdução

## Contexto

Este é um conjunto de dados público brasileiro de comércio eletrónico relativo a encomendas efetuadas na Olist Store. O conjunto de dados contém informações sobre 100 mil encomendas, de 2016 a 2018, efetuadas em várias plataformas de comércio eletrónico no Brasil. As suas características permitem analisar uma encomenda sob várias perspetivas: desde o estado da encomenda, preço, pagamento e desempenho do envio até à localização do cliente, atributos do produto e, por fim, comentários escritos pelos clientes.

Este conjunto de dados foi gentilmente cedido pela Olist, a maior loja de departamentos dos mercados online brasileiros. A Olist conecta pequenas empresas de todo o Brasil a canais de venda de forma simples e com um único contrato. Esses comerciantes podem vender os seus produtos através da Loja Olist e enviá-los diretamente aos clientes através dos parceiros de logística da Olist.

Depois de um cliente adquirir o produto na Olist Store, o vendedor é notificado para processar a encomenda. Assim que o cliente receber o produto, ou quando a data de entrega prevista chegar, o cliente recebe um questionário de satisfação por e-mail, onde pode avaliar a experiência de compra e deixar alguns comentários.

## Dados

O dataset é composto por 9 tabelas relacionadas entre si através de chaves como `order_id`, `customer_id`, `seller_id` e `product_id`:

| Ficheiro | Descrição |
|---|---|
| `olist_orders_dataset.csv` | Tabela central — estado da encomenda, timestamps de compra, aprovação, envio e entrega |
| `olist_order_items_dataset.csv` | Itens por encomenda — produto, vendedor, preço e frete |
| `olist_order_payments_dataset.csv` | Pagamentos — método, parcelas e valor |
| `olist_order_reviews_dataset.csv` | Avaliações — nota (1–5) e comentários do cliente |
| `olist_customers_dataset.csv` | Cliente — cidade, estado e CEP |
| `olist_sellers_dataset.csv` | Vendedor — cidade e estado |
| `olist_products_dataset.csv` | Produto — categoria, dimensões e peso |
| `olist_geolocation_dataset.csv` | Coordenadas geográficas por CEP |
| `product_category_name_translation.csv` | Tradução das categorias PT → EN |

## Objetivo

Este projecto analisa o impacto dos atrasos nas encomendas sobre a experiência do cliente, respondendo a quatro questões:

1. **Assimetria do atraso** — Quais categorias e faixas de valor concentram os maiores atrasos? O atraso distribui-se de forma uniforme ou há padrões específicos?
2. **Ponto de ruptura** — A partir de quantos dias de atraso a avaliação do cliente colapsa? A relação é linear ou existe um limiar crítico?
3. **Logística vs. promessa regional** — Os atrasos resultam de logística lenta ou de janelas de entrega irrealistas? A resposta varia por estado?
4. **Variância por vendedor** — Dentro do mesmo corredor logístico, o tempo de despacho do vendedor é uma variável oculta que explica diferenças de desempenho?

A variável central de análise é `delta_dias`, definida como a diferença entre a data de entrega real e a data de entrega estimada. Valores positivos indicam atraso; valores negativos indicam entrega antecipada.

## Esquema de Dados

![Schema do Dataset Olist](assets/schema.png)

# Setup e Dados

## Ambiente

### Importação bibliotecas

In [2]:
import pandas as pd
#import seaborn as sns
#import matplotlib.pyplot as plt

### Configurações Globais

In [3]:
# Reprodutibilidade — seed único
RANDOM_STATE = 42

# Exibição de dados
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Estilo visual
#sns.set_theme(style='whitegrid', palette='muted')
#plt.rcParams['figure.figsize'] = (10, 6)

### Carregamento de dados

In [4]:
customers_raw = pd.read_csv('../data/raw/olist_customers_dataset.csv')

In [5]:
geolocations_raw = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')

In [6]:
order_items_raw = pd.read_csv('../data/raw/olist_order_items_dataset.csv')

In [7]:
order_payments_raw = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')

In [37]:
order_reviews_raw = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')

In [38]:
orders_raw = pd.read_csv('../data/raw/olist_orders_dataset.csv')      

In [10]:
products_raw = pd.read_csv('../data/raw/olist_products_dataset.csv')

In [11]:
sellers_raw = pd.read_csv('../data/raw/olist_sellers_dataset.csv')

In [12]:
product_category_raw = pd.read_csv('../data/raw/product_category_name_translation.csv')

## Pré Processamento

### Primeiras impressões

#### Customers

In [13]:
# primeira visualização
print(customers_raw.head())

                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  


In [14]:
# descoberta informações sobre os dados  
print(customers_raw.info())

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB
None


Não foram detectados valores nulos no dataframe. 

`customer_zip_code_prefix` que, por ser um identificador geográfico e não um valor numérico, pode ser convertido para `str` por consistência.

#### Geolocation

In [15]:
# primeira visualização
print(geolocations_raw.head())

   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1037           -23.55           -46.64   
1                         1046           -23.55           -46.64   
2                         1046           -23.55           -46.64   
3                         1041           -23.54           -46.64   
4                         1035           -23.54           -46.64   

  geolocation_city geolocation_state  
0        sao paulo                SP  
1        sao paulo                SP  
2        sao paulo                SP  
3        sao paulo                SP  
4        sao paulo                SP  


In [16]:
# descoberta informações sobre os dados  
print(geolocations_raw.info())

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 38.2 MB
None


Não foram detectados valores nulos no dataframe. 

`geolocation_zip_code_prefix` será convertido para `str` pelo mesmo motivo que `customer_zip_code_prefix` — é um identificador geográfico, não um valor numérico. 

A conversão garante também consistência nos joins entre as duas tabelas.

#### Order Items

In [44]:
# primeira visualização
print(order_items_raw.head())

                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   
3  00024acbcdf0a6daa1e931b038114c75              1   
4  00042b26cf59d7ce69dfabb4e55b4fd9              1   

                         product_id                         seller_id  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36   
2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d   
3  7634da152a4610f1595efa32f14722fc  9d7a1d34a5052409006425275ba1c2b4   
4  ac6c3623068f30de03045865e4e10089  df560393f3a51e74553ab94004ba5c87   

   shipping_limit_date  price  freight_value  
0  2017-09-19 09:45:35  58.90          13.29  
1  2017-05-03 11:05:13 239.90          19.93  
2  2018-01-18 14:48:30 199.00          17.87  
3  2018-08-15 10:10:18  12.99          12.79  
4

In [45]:
# descoberta informações sobre os dados  
print(order_items_raw.info())

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB
None


Não foram detectados valores nulos no dataframe.  

`shipping_limit_date` será convertido para `datetime` para permitir operações temporais. 

#### Payments

In [34]:
# primeira visualização
print(order_payments_raw.head())

                           order_id  payment_sequential payment_type  \
0  b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card   
1  a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card   
2  25e8ea4e93396b6fa0d3dd708e76c1bd                   1  credit_card   
3  ba78997921bbcdc1373bb41e913ab953                   1  credit_card   
4  42fdf880ba16b47b59251dd489d4441a                   1  credit_card   

   payment_installments  payment_value  
0                     8          99.33  
1                     1          24.39  
2                     1          65.71  
3                     8         107.78  
4                     2         128.45  


In [35]:
# descoberta informações sobre os dados  
print(order_payments_raw.info())

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB
None


Não foram detectados valores nulos no dataframe. 

Os tipos de dados estão adequados para efeitos de análise.

#### Reviews

In [39]:
# primeira visualização
print(order_reviews_raw.head())

                          review_id                          order_id  \
0  7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb   
1  80e641a11e56f04c1ad469d5645fdfde  a548910a1c6147796b98fdf73dbeba33   
2  228ce5500dc1d8e020d8d1322874b6f0  f9e4b658b201a9f2ecdecbb34bed034b   
3  e64fb393e7b32834bb789ff8bb30750e  658677c97b385a9be170737859d3511b   
4  f7c4243c7fe1938f181bec41a392bdeb  8e6bfb81e283fa7e4f11123a3fb894f1   

   review_score review_comment_title  \
0             4                  NaN   
1             5                  NaN   
2             5                  NaN   
3             5                  NaN   
4             5                  NaN   

                              review_comment_message review_creation_date  \
0                                                NaN  2018-01-18 00:00:00   
1                                                NaN  2018-03-10 00:00:00   
2                                                NaN  2018-02-17 00:00:00   
3           

In [40]:
# descoberta informações sobre os dados  
print(order_reviews_raw.info())

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB
None


Foram detectados valores nulos em `review_comment_title` (87656 registros) e `review_comment_message` (58247 registros). 
Estes valores ausentes são esperados — o cliente pode submeter uma avaliação numérica sem redigir comentário.

`review_creation_date` e `review_answer_timestamp` serão convertidos para `datetime` para permitir operações temporais.

#### Orders

In [41]:
# primeira visualização
print(orders_raw.head())

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08

In [42]:
# descoberta informações sobre os dados  
print(orders_raw.info())

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB
None


Foram detectados valores nulos em `order_approved_at` (160 registos), `order_delivered_carrier_date` (1783 registos) e `order_delivered_customer_date` (2965 registos). 
Estes valores ausentes são esperados — encomendas não concluídas não possuem datas de entrega preenchidas.

`order_purchase_timestamp`, `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date` e `order_estimated_delivery_date` serão convertidos para `datetime` para permitir operações temporais.

#### Products

In [46]:
# primeira visualização
print(products_raw.head())

                         product_id  product_category_name  \
0  1e9e8ef04dbcff4541ed26657ea517e5             perfumaria   
1  3aa071139cb16b67ca9e5dea641aaa2f                  artes   
2  96bd76ec8810374ed1b65e291975717f          esporte_lazer   
3  cef67bcfe19066a932b7673e239eb23d                  bebes   
4  9dc1a7de274444849c219cff195d0b71  utilidades_domesticas   

   product_name_lenght  product_description_lenght  product_photos_qty  \
0                40.00                      287.00                1.00   
1                44.00                      276.00                1.00   
2                46.00                      250.00                1.00   
3                27.00                      261.00                1.00   
4                37.00                      402.00                4.00   

   product_weight_g  product_length_cm  product_height_cm  product_width_cm  
0            225.00              16.00              10.00             14.00  
1           1000.00       

In [47]:
# descoberta informações sobre os dados  
print(products_raw.info())

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB
None


Foram detectados valores nulos em `product_category_name`, `product_name_lenght`, ` product_description_lenght` e `product_photos_qty` (610 registos cada). 
E também e em `product_weight_g`, `product_length_cm`, `product_height_cm` e `product_width_cm` (2 registos cada).

`product_name_lenght`, `product_description_lenght` e `product_photos_qty` aparecem como `float64` devido à presença de nulos — representam contagens inteiras e poderão ser convertidos para `Int64` após tratamento dos valores em falta.

#### Sellers

In [48]:
# primeira visualização.
print(sellers_raw.head())

                          seller_id  seller_zip_code_prefix  \
0  3442f8959a84dea7ee197c632cb2df15                   13023   
1  d1b65fc7debc3361ea86b5f14c68d2e2                   13844   
2  ce3ad9de960102d0677a81f5d0bb7b2d                   20031   
3  c0f3eea2e14555b6faeea3dd58c1b1c3                    4195   
4  51a04a8a6bdcb23deccc82b0b80742cf                   12914   

         seller_city seller_state  
0           campinas           SP  
1         mogi guacu           SP  
2     rio de janeiro           RJ  
3          sao paulo           SP  
4  braganca paulista           SP  


In [49]:
# descoberta informações sobre os dados  
print(sellers_raw.info())

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB
None


Não foram detectados valores nulos no dataframe. 

`seller_zip_code_prefix` será convertido para `str`, seguindo padrões de consistência.

#### Products Category

In [52]:
# primeira visualização
print(product_category_raw.head())

    product_category_name product_category_name_english
0            beleza_saude                 health_beauty
1  informatica_acessorios         computers_accessories
2              automotivo                          auto
3         cama_mesa_banho                bed_bath_table
4        moveis_decoracao               furniture_decor


In [53]:
# descoberta informações sobre os dados  
print(product_category_raw.info())

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB
None


Não foram detectados valores nulos no dataframe. 

Os tipos de dados estão adequados para efeitos de análise.

### Tratamento de Valores Duplicados

In [ ]:
# criação de cópias
customers = customers_raw.copy()
geolocations = geolocations_raw.copy()
order_items = order_items_raw.copy()
order_payments = order_payments_raw.copy()
order_reviews = order_reviews_raw.copy()
orders = orders_raw.copy()
products = products_raw.copy()
sellers = sellers_raw.copy()
product_category = product_category_raw.copy()

In [ ]:
# descoberta de valores duplicados
print(f'Valores duplicados em customers: {customers_raw.duplicated().sum()}')
print(f'Valores duplicados em geolocations: {geolocations_raw.duplicated().sum()}')
print(f'Valores duplicados em order_items: {order_items_raw.duplicated().sum()}')
print(f'Valores duplicados em order_payments: {order_payments_raw.duplicated().sum()}')
print(f'Valores duplicados em order_reviews: {order_reviews_raw.duplicated().sum()}')
print(f'Valores duplicados em orders: {orders_raw.duplicated().sum()}')
print(f'Valores duplicados em products: {products_raw.duplicated().sum()}')
print(f'Valores duplicados em sellers: {sellers_raw.duplicated().sum()}')
print(f'Valores duplicados em product_category: {product_category_raw.duplicated().sum()}')

Valores duplicados em customers: 0
Valores duplicados em geolocations: 261831
Valores duplicados em order_items: 0
Valores duplicados em order_payments: 0
Valores duplicados em order_reviews: 0
Valores duplicados em orders: 0
Valores duplicados em products: 0
Valores duplicados em sellers: 0
Valores duplicados em product_category: 0


In [66]:
# visualização de registros duplicados
geolocations_raw[geolocations_raw.duplicated()]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
15,1046,-23.55,-46.64,sao paulo,SP
44,1046,-23.55,-46.64,sao paulo,SP
65,1046,-23.55,-46.64,sao paulo,SP
66,1009,-23.55,-46.64,sao paulo,SP
67,1046,-23.55,-46.64,sao paulo,SP
...,...,...,...,...,...
1000153,99970,-28.34,-51.87,ciriaco,RS
1000154,99950,-28.07,-52.01,tapejara,RS
1000159,99900,-27.88,-52.22,getulio vargas,RS
1000160,99950,-28.07,-52.01,tapejara,RS


Foram detectados 261.831 valores duplicados em `geolocations`. Este comportamento é esperado — um mesmo CEP pode ter múltiplas entradas com coordenadas ligeiramente distintas. 

Os duplicados serão removidos com base em `geolocation_zip_code_prefix`, mantendo a primeira ocorrência.

In [70]:
# eliminação de registros duplicados em geolocations baseados em geolocation_zip_code_prefix
geolocations = geolocations_raw.drop_duplicates(
    subset=['geolocation_zip_code_prefix']
).reset_index(drop=True)

### Tratamento de Valores Nulos

### Alterações de tipos de dados